In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ts-forecasting/train.parquet
/kaggle/input/competitions/ts-forecasting/test.parquet


# First cell — Imports - You can modify


In [2]:
# Duration: ~8 sec
import gc
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd


def _iter_batches(values, batch_size):
    if batch_size <= 0:
        yield list(values)
        return
    for i in range(0, len(values), batch_size):
        yield list(values[i:i + batch_size])


def downcast_base_features(train_df, test_df, base_feature_cols):
    for col in base_feature_cols:
        if col in train_df.columns:
            train_df[col] = pd.to_numeric(train_df[col], errors='coerce').astype('float32')
        if col in test_df.columns:
            test_df[col] = pd.to_numeric(test_df[col], errors='coerce').astype('float32')


def _compute_feature_block(sorted_batch, group_keys, lag_steps, rolling_windows):
    grouped = sorted_batch.groupby(group_keys, sort=False, observed=True)
    parts = []

    for lag in lag_steps:
        lagged = grouped.shift(lag)
        lagged.columns = [f"{c}_lag_{lag}" for c in sorted_batch.columns]
        parts.append(lagged)

    shifted = grouped.shift(1)
    shifted_grouped = shifted.groupby(group_keys, sort=False, observed=True)

    for name, window in rolling_windows.items():
        if name == 'rolling_std_7':
            rolled = (
                shifted_grouped.rolling(window, min_periods=window)
                .std()
                .reset_index(level=[0, 1], drop=True)
            )
        else:
            rolled = (
                shifted_grouped.rolling(window, min_periods=window)
                .mean()
                .reset_index(level=[0, 1], drop=True)
            )
        rolled.columns = [f"{c}_{name}" for c in sorted_batch.columns]
        parts.append(rolled)

    return pd.concat(parts, axis=1).astype('float32')


def add_ts_features_train(train_df, base_feature_cols, group_cols, ts_col, lag_steps, rolling_windows, batch_size):
    sort_cols = list(group_cols) + [ts_col]
    sorted_idx = train_df.sort_values(sort_cols, kind='mergesort').index
    group_keys = [train_df.loc[sorted_idx, c] for c in group_cols]

    for batch in _iter_batches(base_feature_cols, batch_size):
        sorted_batch = train_df.loc[sorted_idx, batch]
        block_sorted = _compute_feature_block(sorted_batch, group_keys, lag_steps, rolling_windows)
        block = block_sorted.reindex(train_df.index)
        train_df = pd.concat([train_df, block], axis=1, copy=False)
        del sorted_batch, block_sorted, block
        gc.collect()

    return train_df


def add_ts_features_test(train_df, test_df, base_feature_cols, group_cols, ts_col, lag_steps, rolling_windows, batch_size, history_len):
    cols_needed = list(dict.fromkeys(list(group_cols) + [ts_col] + list(base_feature_cols)))

    train_tail = (
        train_df.loc[:, cols_needed]
        .sort_values(list(group_cols) + [ts_col], kind='mergesort')
        .groupby(list(group_cols), sort=False, observed=True)
        .tail(history_len)
        .copy()
    )
    train_tail['__is_test'] = 0
    train_tail['__row_id'] = -1

    test_work = test_df.loc[:, cols_needed].copy()
    test_work['__is_test'] = 1
    test_work['__row_id'] = np.arange(len(test_work), dtype=np.int64)

    combined = pd.concat([train_tail, test_work], axis=0, ignore_index=True, copy=False)
    sorted_idx = combined.sort_values(
        list(group_cols) + ['__is_test', ts_col, '__row_id'], kind='mergesort'
    ).index
    group_keys = [combined.loc[sorted_idx, c] for c in group_cols]

    test_mask = combined['__is_test'].eq(1)
    test_row_ids = combined.loc[test_mask, '__row_id'].to_numpy()

    for batch in _iter_batches(base_feature_cols, batch_size):
        sorted_batch = combined.loc[sorted_idx, batch]
        block_sorted = _compute_feature_block(sorted_batch, group_keys, lag_steps, rolling_windows)
        block = block_sorted.reindex(combined.index)

        test_block = block.loc[test_mask].copy()
        test_block.index = test_row_ids
        test_block = test_block.sort_index(kind='mergesort')
        test_block.index = test_df.index

        test_df = pd.concat([test_df, test_block], axis=1, copy=False)

        del sorted_batch, block_sorted, block, test_block
        gc.collect()

    del train_tail, test_work, combined
    gc.collect()

    return test_df


def fit_safe_label_mappings(df, cols):
    mappings = {}
    for col in cols:
        if col not in df.columns:
            continue
        vals = pd.Series(df[col].dropna().unique())
        try:
            vals = vals.sort_values(ignore_index=True)
        except TypeError:
            vals = vals.reset_index(drop=True)
        mappings[col] = {v: int(i) for i, v in enumerate(vals)}
    return mappings


def apply_safe_label_mappings(df, mappings):
    for col, mapping in mappings.items():
        if col in df.columns:
            df[col] = df[col].map(lambda x: mapping.get(x, -1)).astype('int32')
    return df


def build_feature_list(frame, feature_prefix, categorical_cols, banned_cols):
    feature_cols = [c for c in frame.columns if c.startswith(feature_prefix)]
    feature_cols.extend([c for c in categorical_cols if c in frame.columns])
    feature_cols = list(dict.fromkeys(feature_cols))
    return [c for c in feature_cols if c not in banned_cols]


# Second cell — Variables - You can modify


In [ ]:
# Duration: ~10 sec

# ── Team info ──────────────────────────────────────────────────
TEAM_NAME = "Ayush Shekhar56"
SCORE = 0.2459  # Your public leaderboard score
EMAIL = "ayush.shekhar56@gmail.com"

# ── Paths (assume only these two files exist) ─────────────────
TRAIN_PATH = Path('train.parquet')
TEST_PATH = Path('test.parquet')

# ── Core columns ───────────────────────────────────────────────
ID_COL = 'id'
TARGET = 'y_target'
WEIGHT_COL = 'weight'
TS_COL = 'ts_index'
GROUP_KEYS = ['code', 'sub_code', 'sub_category', 'horizon']
TS_GROUP_COLS = ['code', 'sub_code']
FEATURE_PREFIX = 'feature_'

# ── Feature engineering config ─────────────────────────────────
LAG_STEPS = [1, 3, 5, 14, 21, 28]
ROLLING_WINDOWS = {
    'rolling_mean_3': 3,
    'rolling_mean_7': 7,
    'rolling_std_7': 7,
}
FEATURE_BATCH_SIZE = 8
MAX_HISTORY = max(max(LAG_STEPS), max(ROLLING_WINDOWS.values()))

# ── Model config (fixed, reproducible) ────────────────────────
MODEL_PARAMS = {
    'objective': 'regression',
    'metric': 'rmse',
    'learning_rate': 0.05,
    'num_leaves': 401,
    'max_depth': 10,
    'min_data_in_leaf': 1059,
    'feature_fraction': 0.877728168328829,
    'bagging_fraction': 0.8954376941646838,
    'bagging_freq': 4,
    'lambda_l1': 0.3505859260071018,
    'lambda_l2': 1.2864868506572908,
    'min_gain_to_split': 0.0692818499178563,
    'verbosity': -1,
    'seed': 42,
    'feature_fraction_seed': 42,
    'bagging_seed': 42,
    'data_random_seed': 42,
}
NUM_BOOST_ROUND = 181

EXCLUDED_ZERO_GAIN_FEATURES = {
    'feature_b', 'feature_f', 'feature_g', 'feature_o', 'feature_aa', 'feature_bu',
    'feature_aa_lag_1', 'feature_aa_lag_3', 'feature_aa_lag_5', 'feature_aa_lag_14',
    'feature_aa_lag_21', 'feature_aa_lag_28', 'feature_aa_rolling_mean_7',
    'feature_aa_rolling_std_7', 'feature_ab_rolling_std_7', 'feature_ak_rolling_std_7',
    'feature_au_rolling_std_7', 'feature_av_rolling_std_7', 'feature_ax_lag_3',
    'feature_b_lag_3', 'feature_bb_lag_3', 'feature_b_lag_5', 'feature_bc_lag_5',
    'feature_b_lag_21', 'feature_b_rolling_mean_3', 'feature_b_rolling_mean_7',
    'feature_ax_rolling_std_7', 'feature_b_rolling_std_7', 'feature_ba_rolling_std_7',
    'feature_bb_rolling_std_7', 'feature_bc_rolling_std_7', 'feature_bi_lag_3',
    'feature_bj_lag_3', 'feature_bi_lag_21', 'feature_bi_lag_28', 'feature_bi_rolling_mean_3',
    'feature_bj_rolling_std_7', 'feature_bs_lag_3', 'feature_bt_lag_3', 'feature_bt_lag_14',
    'feature_bw_lag_1', 'feature_c_lag_1', 'feature_bu_lag_3', 'feature_bx_lag_3',
    'feature_bx_lag_14', 'feature_c_lag_14', 'feature_bv_lag_21', 'feature_bv_lag_28',
    'feature_bx_lag_28', 'feature_c_lag_28', 'feature_bu_rolling_mean_3',
    'feature_bw_rolling_mean_7', 'feature_c_rolling_mean_7', 'feature_bv_rolling_std_7',
    'feature_c_rolling_std_7', 'feature_d_lag_1', 'feature_ch_lag_3', 'feature_d_lag_3',
    'feature_d_lag_14', 'feature_d_lag_21', 'feature_d_rolling_mean_7',
    'feature_ch_rolling_std_7', 'feature_d_rolling_std_7', 'feature_f_lag_1', 'feature_g_lag_1',
    'feature_f_lag_3', 'feature_g_lag_3', 'feature_e_lag_5', 'feature_f_lag_5',
    'feature_f_lag_14', 'feature_g_lag_14', 'feature_e_lag_21', 'feature_f_lag_28',
    'feature_g_lag_28', 'feature_f_rolling_mean_3', 'feature_g_rolling_mean_3',
    'feature_e_rolling_mean_7', 'feature_f_rolling_mean_7', 'feature_g_rolling_std_7',
    'feature_o_lag_1', 'feature_o_lag_3', 'feature_o_lag_5', 'feature_s_lag_5',
    'feature_o_lag_14', 'feature_s_lag_14', 'feature_o_lag_28', 'feature_o_rolling_mean_3',
    'feature_p_rolling_std_7', 'feature_q_rolling_std_7', 'feature_w_lag_1',
    'feature_w_lag_5', 'feature_w_lag_28', 'feature_u_rolling_std_7'
}

# ── Load data ──────────────────────────────────────────────────
if not TRAIN_PATH.exists() or not TEST_PATH.exists():
    raise FileNotFoundError('Expected train.parquet and test.parquet in the working directory.')

df = pd.read_parquet(TRAIN_PATH)
test = pd.read_parquet(TEST_PATH)

# Keep row identity stable for prediction cache lookup.
df['__dataset_id'] = 'train'
test['__dataset_id'] = 'test'

CATEGORICAL_COLS = [c for c in GROUP_KEYS if c in df.columns]
BASE_FEATURE_COLS = sorted([
    c for c in df.columns
    if c.startswith(FEATURE_PREFIX) and '_lag_' not in c and '_rolling_' not in c
])

if not BASE_FEATURE_COLS:
    raise ValueError('No base feature columns found (expected feature_* columns).')

# EDA feature subset only (training uses full engineered set).
feature = BASE_FEATURE_COLS[: min(8, len(BASE_FEATURE_COLS))]

# Globals populated in model-fit cell.
LABEL_MAPPINGS = {}
MODEL_FEATURES = []
TRAIN_BASE_HISTORY = None
PRED_CACHE = {}


# Test - Do not modify

In [4]:
# ── Validation checks ──────────────────────────────────────────

if TEAM_NAME is None or TEAM_NAME == "A data COMPANY":
    print("❌ You did not fill in the TEAM_NAME.")

if EMAIL is None or EMAIL == "yourname@email.com":
    print("❌ You did not fill in the EMAIL.")

if SCORE is None:
    print("❌ You did not fill in the SCORE.")

if not isinstance(SCORE, float):
    print(f"❌ SCORE must be a float (e.g. 0.25), got {type(SCORE)}.")

if not (0.0 <= SCORE <= 1.0):
    print(f"❌ SCORE must be between 0.0 and 1.0, got {SCORE}.")

❌ You did not fill in the TEAM_NAME.
❌ You did not fill in the EMAIL.


# Third cell — EDA - You can modify

⚠️ One EDA cell only — keep it minimal and relevant.

In [5]:
# Duration: ~20 sec

print(f"train shape: {df.shape}, test shape: {test.shape}")
print(f"ts range train: {df[TS_COL].min()} -> {df[TS_COL].max()}")

horizon_stats = (
    df.groupby('horizon', observed=True)[TARGET]
    .agg(['count', 'mean', 'std'])
    .reset_index()
)
horizon_stats.head(10)


y_target                                                              \
           count      mean       std       min       25%       50%       75%   
horizon                                                                        
1           25.0 -0.021023  0.266384 -0.445647 -0.140062 -0.033479  0.087466   
3           25.0 -0.067018  0.365120 -0.912869 -0.195680 -0.018421  0.087887   
10          25.0 -0.439052  0.825096 -1.849698 -0.804196 -0.242618  0.112842   
25          25.0 -2.235528  0.899368 -3.577121 -2.830697 -2.579381 -1.743121   

                  feature_f             ... feature_u           feature_v  \
              max     count       mean  ...       75%       max     count   
horizon                                 ...                                 
1        0.558906      25.0   8.135533  ...  0.216898  0.285737      25.0   
3        0.581993      25.0  10.031240  ...  0.216898  0.285737      25.0   
10       0.659961      25.0   8.486522  ...  0.216898  0.285737      25.0   
25      -0.437398      25.0   8.125786  ...  0.216898  0.285737      25.0   

                                                                               
             mean       std       min       25%       50%       75%       max  
horizon                                                                        
1        0.098264  0.004263  0.091141  0.094203  0.097964  0.101463  0.103819  
3        0.098264  0.004263  0.091141  0.094203  0.097964  0.101463  0.103819  
10       0.098264  0.004263  0.091141  0.094203  0.097964  0.101463  0.103819  
25       0.098264  0.004263  0.091141  0.094203  0.097964  0.101463  0.103819  

[4 rows x 144 columns]

# Fourth cell — Functions 
- Do **NOT** modify: `pred, weighted_rmse_score, _clip01`
- **Modify**: `feature_eng_and_predict`


In [6]:
# Duration: ~5 sec

def feature_eng_and_predict(X_cut, model):
    # Feature engineering for prediction without forward-looking leakage.
    if X_cut.empty:
        return np.array([], dtype=np.float64)

    dataset_id = str(X_cut['__dataset_id'].iloc[0]) if '__dataset_id' in X_cut.columns else 'adhoc'
    cache_key = (dataset_id, id(model))

    if cache_key not in PRED_CACHE:
        if dataset_id == 'test':
            full_frame = test.copy()
            engineered = add_ts_features_test(
                train_df=TRAIN_BASE_HISTORY,
                test_df=full_frame,
                base_feature_cols=BASE_FEATURE_COLS,
                group_cols=TS_GROUP_COLS,
                ts_col=TS_COL,
                lag_steps=LAG_STEPS,
                rolling_windows=ROLLING_WINDOWS,
                batch_size=FEATURE_BATCH_SIZE,
                history_len=MAX_HISTORY,
            )
        elif dataset_id == 'train':
            full_frame = df.copy()
            engineered = add_ts_features_train(
                train_df=full_frame,
                base_feature_cols=BASE_FEATURE_COLS,
                group_cols=TS_GROUP_COLS,
                ts_col=TS_COL,
                lag_steps=LAG_STEPS,
                rolling_windows=ROLLING_WINDOWS,
                batch_size=FEATURE_BATCH_SIZE,
            )
        else:
            full_frame = X_cut.copy()
            engineered = add_ts_features_train(
                train_df=full_frame,
                base_feature_cols=BASE_FEATURE_COLS,
                group_cols=TS_GROUP_COLS,
                ts_col=TS_COL,
                lag_steps=LAG_STEPS,
                rolling_windows=ROLLING_WINDOWS,
                batch_size=FEATURE_BATCH_SIZE,
            )

        engineered = apply_safe_label_mappings(engineered, LABEL_MAPPINGS)
        for col in MODEL_FEATURES:
            if col not in engineered.columns:
                engineered[col] = np.nan

        pred_values = model.predict(engineered[MODEL_FEATURES], num_iteration=model.current_iteration())
        PRED_CACHE[cache_key] = pd.Series(pred_values, index=engineered.index)

    return PRED_CACHE[cache_key].reindex(X_cut.index).to_numpy()

def _clip01(x: float) -> float:
    return float(np.minimum(np.maximum(x, 0.0), 1.0))

def weighted_rmse_score(y_target, y_pred, w) -> float:
    denom = np.sum(w * y_target ** 2)
    ratio = np.sum(w * (y_target - y_pred) ** 2) / denom
    clipped = _clip01(ratio)
    val = 1.0 - clipped
    return float(np.sqrt(val))

def pred(X, ts_index, model):
    mask    = X["ts_index"] <= ts_index
    X_cut   = X[mask]
    y = feature_eng_and_predict(X_cut, model)
    mask_ts = (X_cut["ts_index"] == ts_index).values
    result  = X_cut[X_cut["ts_index"] == ts_index][GROUP_KEYS + ["ts_index"]].copy()
    result["pred"] = y[mask_ts]
    return result


# Fifth cell — Model fit - You can modify
- don't forget to implement the logic in the `feature_eng_and_predict` function
- do not include forward looking data in df, i.e. data at a ts_index that includes information for a ts_index in the future (e.g. global average with test data)


In [7]:
# Duration: ~4 min
# a. Feature selection & engineering here
# b. Only fit the model used for prediction, not your model selection & engineering
# ⚠️ Never use the test dataset here

df = df.dropna(subset=[TARGET]).copy()
df = df.sort_values(TS_GROUP_COLS + [TS_COL], kind='mergesort').reset_index(drop=True)
test = test.sort_values(TS_GROUP_COLS + [TS_COL], kind='mergesort').reset_index(drop=True)

downcast_base_features(df, test, BASE_FEATURE_COLS)

TRAIN_BASE_HISTORY = df[TS_GROUP_COLS + [TS_COL] + BASE_FEATURE_COLS].copy()

train_model_df = add_ts_features_train(
    train_df=df.copy(),
    base_feature_cols=BASE_FEATURE_COLS,
    group_cols=TS_GROUP_COLS,
    ts_col=TS_COL,
    lag_steps=LAG_STEPS,
    rolling_windows=ROLLING_WINDOWS,
    batch_size=FEATURE_BATCH_SIZE,
)

LABEL_MAPPINGS = fit_safe_label_mappings(train_model_df, CATEGORICAL_COLS)
train_model_df = apply_safe_label_mappings(train_model_df, LABEL_MAPPINGS)

auto_features = build_feature_list(
    frame=train_model_df,
    feature_prefix=FEATURE_PREFIX,
    categorical_cols=CATEGORICAL_COLS,
    banned_cols={TARGET, WEIGHT_COL, TS_COL, ID_COL, '__dataset_id'},
)
MODEL_FEATURES = [c for c in auto_features if c not in EXCLUDED_ZERO_GAIN_FEATURES]

if not MODEL_FEATURES:
    raise ValueError('No model features left after applying excluded feature list.')

weights = (
    train_model_df[WEIGHT_COL].to_numpy(copy=False)
    if WEIGHT_COL in train_model_df.columns
    else np.ones(len(train_model_df), dtype=np.float64)
)

train_set = lgb.Dataset(
    train_model_df[MODEL_FEATURES],
    label=train_model_df[TARGET],
    weight=weights,
    categorical_feature=[c for c in CATEGORICAL_COLS if c in MODEL_FEATURES],
    free_raw_data=False,
)

# do not change the 'model' variable name, used in the next cell
model = lgb.train(
    params=MODEL_PARAMS,
    train_set=train_set,
    num_boost_round=NUM_BOOST_ROUND,
)

PRED_CACHE = {}

print(f"n_base_features={len(BASE_FEATURE_COLS)}")
print(f"n_model_features={len(MODEL_FEATURES)}")


LinearRegression()

# Sixth cell —  Train Score - Do not modify

In [8]:
# Duration: ~XX min

# Sequential prediction to ensure no forward looking information is used
all_results = []
for ts in sorted(df["ts_index"].unique()):
    result = pred(df, ts, model)
    all_results.append(result)

train_preds = pd.concat(all_results, ignore_index=True)

# Merge with ground truth
train_preds = train_preds.merge(
    df['y_target'].rename("y_true").reset_index(),
    left_index=True,
    right_index=True,
    how="left"
)

# Compute score (weights = 1 if no weight column available)
w = np.ones(len(train_preds))  # replace with actual weights if available
train_score = weighted_rmse_score(
    y_target = train_preds["y_true"].values,
    y_pred   = train_preds["pred"].values,
    w        = w
)

print(f"Train weighted RMSE score: {train_score:.6f}")

Train weighted RMSE score: 0.608956


# Seventh cell — Prediction - Do not modify


In [9]:
# Duration: ~XX min

all_results = []
for ts in sorted(test["ts_index"].unique()):
    all_results.append(pred(test, ts, model))
predictions_df = pd.concat(all_results, ignore_index=True)
predictions_df["id"] = (
    predictions_df["code"]                + "__" +
    predictions_df["sub_code"]            + "__" +
    predictions_df["sub_category"]        + "__" +
    predictions_df["horizon"].astype(str) + "__" +
    predictions_df["ts_index"].astype(str)
)
predictions_df = predictions_df[["id", "pred"]].rename(columns={"pred": "prediction"})[['id', 'prediction']]

predictions_df.to_csv("submission.csv", index=False, sep=";")

In [10]:
predictions_df

,id,prediction
0,W2MW3G2L__495MGHFJ__PZ9S1Z4V__3__3647,-125.682899
1,W2MW3G2L__495MGHFJ__PZ9S1Z4V__10__3647,-125.474374
2,W2MW3G2L__495MGHFJ__PZ9S1Z4V__25__3647,-125.110213
3,W2MW3G2L__495MGHFJ__PZ9S1Z4V__1__3647,-125.786554
4,W2MW3G2L__495MGHFJ__PZ9S1Z4V__10__3648,-124.415285
...,...,...
95,W2MW3G2L__495MGHFJ__PZ9S1Z4V__10__3670,-113.925511
96,W2MW3G2L__495MGHFJ__PZ9S1Z4V__25__3671,-114.084488
97,W2MW3G2L__495MGHFJ__PZ9S1Z4V__3__3671,-114.323303
98,W2MW3G2L__495MGHFJ__PZ9S1Z4V__1__3671,-114.065224
